# A FAIR² Mental Health Survey Dataset Exploration with `mlcroissant`
This notebook guides you through loading and exploring the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json) using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The steps include:
- Loading the dataset using the Croissant schema URL
- Discovering available record sets, fields, and columns (referenced by their `@id`)
- Extracting tabular data for analysis
- Performing basic exploratory analysis and visualizations

### Dataset Source
The dataset's schema is provided by a Croissant JSON-LD file at the following URL:

In [ ]:
# Ensure the `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and explore high-level information about the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print general metadata
meta_json = dataset.metadata.to_json()
print(f"Title: {meta_json['name']}")
print(f"Version: {meta_json.get('version', 'N/A')}")
print(f"Published: {meta_json.get('datePublished', 'N/A')}")
print(f"Identifier: {meta_json.get('identifier', 'N/A')}")
print(f"Description: {meta_json['description']}")
print(f"Keywords: {', '.join(meta_json.get('keywords', []))}")

## 2. Data Overview
List available record sets, fields, and their `@id` values. All further steps will reference these by their `@id` for reproducibility and clarity.

In [ ]:
# List all record sets and their ID
print("Available Record Sets (by @id):")
for rs in dataset.record_sets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")
    # For each record set, list its fields
    if 'field' in rs:
        print("  Fields (@id):")
        for field in rs['field']:
            print(f"    - {field['@id']}: {field.get('name', '')} ({field.get('dataType', '')})")
    # List columns
    if 'column' in rs:
        print("  Columns (@id):")
        for col in rs['column']:
            print(f"    - {col['@id']}: {col.get('name', '')} ({col.get('dataType', '')})")

## 3. Data Extraction
Load available record sets into pandas DataFrames for analysis. **You must reference by `@id`.**

Below, we demonstrate loading all available record sets into a dictionary of DataFrames. Adjust the list `record_set_ids` according to the printed `@id`s above.

In [ ]:
# Collect all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    # The mlcroissant library returns an iterator over records
    records = list(dataset.records(record_set=rs_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded record set '{rs_id}' with {len(df)} records. Columns:")
        print(df.columns.tolist())
        print(df.head(2))
    else:
        print(f"[!] Record set '{rs_id}' has no records.")

## 4. Exploratory Data Analysis (EDA)
Perform simple statistics and data filtering on the main survey record set. This example assumes a main record set includes numerical mental health survey scores, e.g., `GAD-7`, `PHQ-9`, `PSQ`. **Adjust the variable names below according to column `@id`s from your data.**

In [ ]:
# For demonstration, let's assume the main data is in the first record set.
# Replace with the actual @id, e.g., main_survey_rs = 'cr:RecordSet-mental_health_survey'

if record_set_ids:
    main_rs_id = record_set_ids[0]
    df = dataframes[main_rs_id]
    print(f"Exploring record set: {main_rs_id}")
    
    # List available columns
    print(f"Available columns: {df.columns.tolist()}")
    
    # Pick a numeric score column, for example 'GAD-7'. Use the actual @id from overview!
    # In practice, these might be 'gad7_score' or similar.
    gad7_field = None
    for col in df.columns:
        if 'gad' in col.lower():
            gad7_field = col
            break
    if gad7_field:
        # Convert to numeric if necessary
        df[gad7_field] = pd.to_numeric(df[gad7_field], errors='coerce')
        # Simple distribution overview
        print(df[gad7_field].describe())
        # Filter for moderate or severe anxiety (e.g., GAD-7 >= 10)
        threshold = 10
        filtered_df = df[df[gad7_field] >= threshold]
        print(f"Records with {gad7_field} >= {threshold}: {len(filtered_df)}")
        # Normalize
        filtered_df[f"{gad7_field}_normalized"] = (filtered_df[gad7_field] - filtered_df[gad7_field].mean()) / filtered_df[gad7_field].std()
        print(filtered_df[[gad7_field, f"{gad7_field}_normalized"]].head())
        # Example grouping: by gender, if available
        group_field = None
        for col in df.columns:
            if 'gender' in col.lower():
                group_field = col
                break
        if group_field:
            grouped = filtered_df.groupby(group_field)[gad7_field].mean()
            print(f"Mean {gad7_field} by {group_field} (for scores >= {threshold}):")
            print(grouped)
    else:
        print("No GAD-7-like field found. Please adjust the analysis to suit available data.")
else:
    print("No record sets loaded.")

## 5. Visualization
Visualize the distribution of a numeric mental health field (e.g., GAD-7) using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and 'df' in locals() and gad7_field:
    plt.figure(figsize=(8,4))
    sns.histplot(df[gad7_field].dropna(), kde=True, bins=15)
    plt.title(f"Distribution of {gad7_field}")
    plt.xlabel(gad7_field)
    plt.ylabel('Count')
    plt.show()
    
    # If gender field exists, show boxplot
    if group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field], y=df[gad7_field])
        plt.title(f"{gad7_field} by {group_field}")
        plt.show()

## 6. Conclusion
- This notebook illustrated loading a Croissant-formatted dataset with `mlcroissant`, referencing all entities by their `@id`.
- You learned how to enumerate record sets, extract data by record set, and perform basic EDA and plotting on survey scores such as GAD-7.
- For further opportunities, adjust the field and record set `@id`s to explore other fields, demographic splits, or perform predictive modeling as suitable for your analysis.